# 🎬 BigByte Systems: Free Cloud GPU Talking Avatar Video Generator
Generate 100% realistic, lip-synced studio podcast interview videos for **Host Ananya** and **Specialist Vikram**.

### ⚡ Zero Cost & Zero Load on Your PC
- **Hardware**: Google's Cloud 16GB NVIDIA Tesla T4 GPU (₹0 cost).
- **Execution Time**: ~2 to 3 minutes.
- **Instruction**: Click **Runtime** ➔ **Run all** (or press `Ctrl + F9`). The finished `.mp4` video files will download to your browser automatically!

In [ ]:
# 1. Clone SadTalker & Install Dependencies
!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!pip install -q torch torchvision torchaudio
!pip install -q -r requirements.txt
!pip install -q imageio-ffmpeg facexlib kornia yacs gfpgan
!apt-get update -qq && apt-get install -y -qq ffmpeg aria2

In [ ]:
# 2. Download ALL Neural Checkpoints (Including BFM_Fitting & Face Enhancer)
!mkdir -p ./checkpoints ./gfpgan/weights

# Main Model & Mapping Networks
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00109-model.pth.tar -d ./checkpoints -o mapping_00109-model.pth.tar
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/mapping_00229-model.pth.tar -d ./checkpoints -o mapping_00229-model.pth.tar
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2-rc/SadTalker_V0.0.2_256.safetensors -d ./checkpoints -o SadTalker_V0.0.2_256.safetensors

# 3D Morphable Model (BFM_Fitting) & Torch Hub Models (Crucial!)
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip -d ./checkpoints -o BFM_Fitting.zip
!unzip -o -q ./checkpoints/BFM_Fitting.zip -d ./checkpoints/

!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/OpenTalker/SadTalker/releases/download/v0.0.2/hub.zip -d ./checkpoints -o hub.zip
!unzip -o -q ./checkpoints/hub.zip -d ./checkpoints/

# Face Enhancer (GFPGAN) Weights
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/xinntao/facexlib/releases/download/v0.1.0/alignment_WFLW_4HG.pth -d ./gfpgan/weights -o alignment_WFLW_4HG.pth
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/xinntao/facexlib/releases/download/v0.1.0/detection_Resnet50_Final.pth -d ./gfpgan/weights -o detection_Resnet50_Final.pth
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth -d ./gfpgan/weights -o GFPGANv1.4.pth
!aria2c --console-log-level=error -c -x 8 -s 8 https://github.com/xinntao/facexlib/releases/download/v0.2.2/parsing_parsenet.pth -d ./gfpgan/weights -o parsing_parsenet.pth

print('✅ All neural checkpoints and 3D face models downloaded successfully!')

In [ ]:
# 3. Fetch Portraits & Neural Speech Audio from your GitHub repo
!wget -q -O ananya.jpg https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/images/interviewer_ananya.jpg
!wget -q -O vikram.jpg https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/images/expert_vikram.jpg
!wget -q -O audio_ananya.mp3 https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/audio/interview_vsl/seg1_ananya.mp3
!wget -q -O audio_vikram.mp3 https://raw.githubusercontent.com/bbsthane-web/bigbyte-hyperlocal/master/assets/audio/interview_vsl/seg1_vikram.mp3

# Convert to 16kHz mono WAV for high-fidelity lip-sync extraction
!ffmpeg -y -i audio_ananya.mp3 -ar 16000 -ac 1 audio_ananya.wav
!ffmpeg -y -i audio_vikram.mp3 -ar 16000 -ac 1 audio_vikram.wav

# 4. Render Lip-Synced Video for Ananya (Host)
print('🎬 [1/2] Generating Ananya Talking Video...')
!python inference.py \
    --driven_audio audio_ananya.wav \
    --source_image ananya.jpg \
    --result_dir ./results_ananya \
    --still \
    --preprocess crop \
    --enhancer gfpgan

# 5. Render Lip-Synced Video for Vikram (Growth Architect)
print('🎬 [2/2] Generating Vikram Talking Video...')
!python inference.py \
    --driven_audio audio_vikram.wav \
    --source_image vikram.jpg \
    --result_dir ./results_vikram \
    --still \
    --preprocess crop \
    --enhancer gfpgan

In [ ]:
# 6. Find generated MP4s inside timestamped subfolders and download
import glob, shutil, os
from google.colab import files

jobs = [
    ('ananya_podcast.mp4', './results_ananya'),
    ('vikram_podcast.mp4', './results_vikram')
]

for filename, folder in jobs:
    # Search recursively for .mp4 in all subfolders
    videos = glob.glob(f'{folder}/**/*.mp4', recursive=True)
    if videos:
        latest = max(videos, key=os.path.getmtime)
        shutil.copy(latest, filename)
        print(f'✅ Found video: {latest}')
        print(f'🚀 Downloading {filename} ({os.path.getsize(filename):,} bytes)...')
        files.download(filename)
    else:
        print(f'⚠️ No video found in {folder}')
        if os.path.exists(folder):
            print(f'Folder listing for {folder}:', os.listdir(folder))

In [ ]:
# 7. Watch Generated Videos Live in Colab!
from IPython.display import HTML, display
from base64 import b64encode

for filename in ['ananya_podcast.mp4', 'vikram_podcast.mp4']:
    if os.path.exists(filename):
        data_url = 'data:video/mp4;base64,' + b64encode(open(filename, 'rb').read()).decode()
        display(HTML(f'''
        <div style="margin: 15px 0;">
            <h4>🎬 {filename}</h4>
            <video width="320" height="320" controls autoplay loop style="border-radius: 12px;">
                <source src="{data_url}" type="video/mp4">
            </video>
        </div>
        '''))